# Optimization Campaign (HITL)

Interactive notebook for running prompt optimization campaigns with full human-in-the-loop control.

**Workflow:** Config → Replay → Diagnostics → Baseline Eval → Optimization Round → LLM Suggestions → Repeat

**Prerequisites:** TermNorm running at `http://127.0.0.1:8000`, Groq API key set in `.env`

## 1. Setup

In [ ]:
#@title Setup & imports
import json
import os

import pandas as pd

from _campaign_lib import (
    init_services, run_or_load_replay, analyze_candidate_coverage,
    load_baseline_prompt, filter_eval_data, evaluate_prompt,
    generate_candidates, select_round_winner, generate_suggestions,
    display_suggestions, save_campaign_winner,
)

svc = init_services()
campaign_rounds = []
print("Ready.")

## 2. Campaign Config

Edit this cell and re-run to change settings for replay, pipeline parameters, optimization, and the evaluation LLM. After each round, the LLM suggestion cell will print a modified config you can copy back here.

In [ ]:
campaign_config = {
    "replay": {
        "skip_llm_ranking": False,
        "query_limit": 0,               # 0=all, N=first N for quick test
        "delay_between": 2.0,
    },
    "pipeline_params": {
        "max_sites": 7,                  # Web: pages fetched
        "num_results": 20,               # Web: search results count
        "content_char_limit": 800,       # Web: chars per page
        "raw_content_limit": 5000,       # LLM1: research text input
        "profiling_temperature": 0.3,    # LLM1: temperature
        "profiling_max_tokens": 1800,    # LLM1: output limit
        "ranking_temperature": 0,        # LLM2: temperature
        "ranking_max_tokens": 4000,      # LLM2: output limit
        "ranking_sample_size": 20,       # LLM2: candidates to rerank
        "max_token_candidates": 20,      # Token matching: kept
        "relevance_weight_core": 0.7,    # Scoring: core vs spec weight
    },
    "optimization": {
        "n_variants": 5,
        "creativity": 0.7,
        "improvement_threshold": 0.01,
        "max_rounds": 3,
    },
    "eval_llm": {
        "model": "meta-llama/llama-4-maverick-17b-128e-instruct",
        "provider_url": "https://api.groq.com/openai/v1/chat/completions",
        "temperature": 0,
        "max_tokens": 4000,
    },
}

print(json.dumps(campaign_config, indent=2))

## 3. Run Replay

Replay queries against TermNorm with the configured settings. Uses cache when pipeline_params haven't changed.

In [ ]:
#@title Replay pipeline
execution, replay_results = await run_or_load_replay(
    svc["client"], svc["store"], svc["queries"], svc["terms"],
    "termnorm-local", "1_production_historical",
    campaign_config["replay"], campaign_config["pipeline_params"],
)

## 4. Diagnostic — Candidate Coverage

For each query: is the ground truth in the token-matched candidates? At what rank? This determines whether reranker optimization is viable (ground truth must be in the candidate set for the reranker to promote it).

In [ ]:
#@title Candidate coverage analysis
cov_df = analyze_candidate_coverage(replay_results)

In [ ]:
#@title Sample entity profiles (qualitative check)
n_samples = 3
samples = [r for r in replay_results if r.get("pipeline_data", {}).get("entity_profile")][:n_samples]

for i, s in enumerate(samples):
    profile = s["pipeline_data"]["entity_profile"]
    print(f"--- Sample {i+1}: {s['query'][:60]} ---")
    print(f"  Core concept: {profile.get('core_concept', '?')}")
    print(f"  Profile keys: {list(profile.keys())}")
    print(f"  Ground truth: {s['ground_truth']}")
    candidates = s.get("pipeline_data", {}).get("token_matched_candidates", [])[:5]
    print(f"  Top 5 candidates: {[c[0] if isinstance(c, (list,tuple)) else c for c in candidates]}")
    print()

## 5. Load Baseline & Evaluate

Load the current `llm_ranking` prompt from the synced experiment, wrap it in a PromptState, and evaluate it locally using cached pipeline data.

In [ ]:
#@title Load baseline & evaluation data
baseline = load_baseline_prompt(svc["exp_data"])
eval_data = filter_eval_data(replay_results)
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

In [ ]:
#@title Evaluate baseline prompt
baseline_results = await evaluate_prompt(
    baseline, eval_data, campaign_config["eval_llm"], GROQ_API_KEY, label="Baseline",
)
baseline_hits = sum(1 for r in baseline_results if r["hit"])
baseline_accuracy = baseline_hits / len(baseline_results) if baseline_results else 0
campaign_rounds = [{
    "round": 0, "label": "baseline", "prompt_state": baseline,
    "accuracy": baseline_accuracy, "hits": baseline_hits,
    "total": len(baseline_results), "results": baseline_results,
}]
failures = [r for r in baseline_results if not r["hit"] and not r["error"]]
for r in failures[:5]:
    print(f"  MISS: {r['query'][:55]}  |  Pred: {r['predicted'][:35]}  |  GT: {r['ground_truth'][:35]}")

## 6. Run One Optimization Round

Analyze failures from the current best, generate N candidate prompts, evaluate all locally, select the round winner. Re-run this section for additional rounds.

In [ ]:
#@title Run optimization round
opt = campaign_config["optimization"]
current_best = campaign_rounds[-1]
print(f"=== ROUND {len(campaign_rounds)} === Current best: {current_best['label']} ({current_best['accuracy']:.1%})\n")

candidates = await generate_candidates(
    current_best["prompt_state"], current_best["accuracy"], current_best["results"],
    opt["n_variants"], opt["creativity"], campaign_config["eval_llm"], GROQ_API_KEY,
)
all_candidate_results = {}
for idx, c in enumerate(candidates):
    all_candidate_results[c.id] = await evaluate_prompt(
        c, eval_data, campaign_config["eval_llm"], GROQ_API_KEY, label=f"Candidate {idx+1}",
    )
round_entry = select_round_winner(candidates, all_candidate_results, current_best, opt["improvement_threshold"])
round_entry["round"] = len(campaign_rounds)
campaign_rounds.append(round_entry)

## 7. LLM Suggestion for Next Round (HITL)

After each round, the LLM analyzes failures and suggests:
1. Failure pattern analysis
2. Parameter change suggestions
3. Prompt phrase fragments to adopt
4. Suggested next `campaign_config`

**Review the suggestions, edit the config cell (Section 2), then re-run Sections 6-7.**

In [ ]:
#@title Generate LLM suggestions for next round
suggestions = await generate_suggestions(
    campaign_rounds, replay_results, campaign_config, campaign_config["eval_llm"], GROQ_API_KEY,
)
display_suggestions(suggestions, len(campaign_rounds))
print(f"\n--- SUGGESTED CONFIG (copy to Section 2) ---")
print(json.dumps(suggestions.get("suggested_config", campaign_config), indent=2))

## 8. Campaign Summary

Compare all rounds, track per-query flips, display the PromptState lineage chain, and save the winner.

In [ ]:
#@title Campaign comparison table
rows = []
for rd in campaign_rounds:
    rows.append({
        "round": rd["round"],
        "label": rd["label"][:40],
        "hit@1": rd["hits"],
        "total": rd["total"],
        "accuracy": f"{rd['accuracy']:.1%}",
        "prompt_id": rd["prompt_state"].id[:12],
    })

print(f"CAMPAIGN SUMMARY ({len(campaign_rounds)} rounds)")
print(f"{'='*70}")
display(pd.DataFrame(rows))

In [ ]:
#@title Per-query flip tracking (baseline vs final)
if len(campaign_rounds) >= 2:
    base_r = campaign_rounds[0]["results"]
    final_r = campaign_rounds[-1]["results"]

    flips = []
    for br, fr in zip(base_r, final_r):
        b_hit = br["hit"]
        f_hit = fr["hit"]
        if b_hit != f_hit:
            flips.append({
                "query": br["query"][:50],
                "flip": "MISS->HIT" if f_hit else "HIT->MISS",
                "base_pred": br["predicted"][:35],
                "final_pred": fr["predicted"][:35],
                "ground_truth": br["ground_truth"][:35],
            })

    gained = sum(1 for f in flips if f["flip"] == "MISS->HIT")
    lost = sum(1 for f in flips if f["flip"] == "HIT->MISS")

    print(f"FLIP TRACKING (baseline -> round {campaign_rounds[-1]['round']})")
    print(f"  Queries gained (MISS->HIT): {gained}")
    print(f"  Queries lost (HIT->MISS):   {lost}")
    print(f"  Net change:                 {gained - lost:+d}")
    print()
    if flips:
        display(pd.DataFrame(flips))
else:
    print("Need at least 2 rounds for flip tracking.")

In [ ]:
#@title PromptState lineage chain
print("LINEAGE CHAIN")
print("="*50)
for i, rd in enumerate(campaign_rounds):
    ps = rd["prompt_state"]
    parent = ps.parent_id[:12] if ps.parent_id else "root"
    arrow = "  " if i == 0 else "  -> "
    print(f"{arrow}[{ps.id[:12]}] Round {rd['round']}: {rd['label'][:40]} ({rd['accuracy']:.1%})")
    if ps.parent_id:
        print(f"       parent: {parent}  |  changes: {ps.changes_description or 'none'}")

In [ ]:
#@title Save winner
save_campaign_winner(campaign_rounds, campaign_config, svc["store"], "termnorm-local")